In [ ]:
import pandas as pd
import numpy as np
X_train = pd.read_csv('data/processed/train_X.csv')
y_train = pd.read_csv('data/processed/train_y.csv')
X_test  = pd.read_csv('data/processed/test_X.csv')

X = X_train.copy()
y = y_train['CS']

In [ ]:
date_cols_to_process = ['X.2', 'X.3', 'X.4', 'X.5', 'X.6']

# Filter for columns that actually exist in the DataFrame X
existing_date_cols_X = [col for col in date_cols_to_process if col in X.columns]
existing_date_cols_X_test = [col for col in date_cols_to_process if col in X_test.columns]

# Process date columns only if they exist
for col in existing_date_cols_X:
    X[col] = pd.to_datetime(X[col], errors='coerce')
    X[col] = X[col].map(lambda x: x.toordinal() if pd.notnull(x) else 0)

for col in existing_date_cols_X_test:
    X_test[col] = pd.to_datetime(X_test[col], errors='coerce')
    X_test[col] = X_test[col].map(lambda x: x.toordinal() if pd.notnull(x) else 0)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
print("Claim rate:", y.mean())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_split, y_train_split)

log_pred = log_model.predict_proba(X_val)[:,1]
print("Logistic AUC:", roc_auc_score(y_val, log_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train_split, y_train_split)

rf_pred = rf.predict_proba(X_val)[:,1]
print("Random Forest AUC:", roc_auc_score(y_val, rf_pred))

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42
)

xgb.fit(X_train_split, y_train_split)

xgb_pred = xgb.predict_proba(X_val)[:,1]
print("XGBoost AUC:", roc_auc_score(y_val, xgb_pred))


In [ ]:
import matplotlib.pyplot as plt

importances = xgb.feature_importances_
feat_names = X.columns

sorted_idx = np.argsort(importances)[-10:]

plt.figure(figsize=(8,5))
plt.barh(range(len(sorted_idx)), importances[sorted_idx])
plt.yticks(range(len(sorted_idx)), feat_names[sorted_idx])
plt.title("Top Features")
plt.show()

In [ ]:
xgb.fit(X, y)

In [ ]:
CS_pred = xgb.predict_proba(X_test)[:,1]

In [ ]:
submission = pd.DataFrame({'CS': CS_pred})
submission.to_csv('CS_predictions.csv', index=False)

In [ ]:
submission.to_csv('outputs/CS_predictions.csv', index=False)

In [ ]:
submission.describe()